# ReAct from scratch

Implements the Reasoning + Acting loop end-to-end without any agent framework:
a strict regex parser for `Thought / Action / Observation` turns, three typed
tools (`calculator`, `wiki_search`, `get_datetime`), a task list of 20
arithmetic / lookup / temporal queries, and a small rule-based policy that
emits well-formed ReAct transcripts without requiring a running LLM. A
``VLLMPolicy`` stub documents how to swap in `Qwen/Qwen2.5-0.5B-Instruct`
locally when a GPU is available.

**Learning objectives**
- Understand ReAct's ``Thought → Action → Observation`` loop and its parser.
- Build safe tool wrappers (no `eval`) and a tool registry with typed stubs.
- Score an agent on success rate, parse-error rate, and average step count.

**Papers**: Yao et al. 2022 *ReAct: Synergizing Reasoning and Acting in Language Models*.
**Hardware**: CPU. GPU optional for the `VLLMPolicy` path.
**Estimated runtime**: ≈30 s.


In [ ]:
from __future__ import annotations

import ast
import operator as op
import re
import sys
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

from llm_infra_lab._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("04_agents_01_react_from_scratch")


In [ ]:
ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*?)\]", re.DOTALL)
FINAL_RE = re.compile(r"Final\s*Answer:\s*(.*)", re.DOTALL)


@dataclass
class ParsedStep:
    thought: str | None
    action: str | None
    action_input: str | None
    final_answer: str | None
    raw: str

    @property
    def is_action(self) -> bool:
        return self.action is not None

    @property
    def is_final(self) -> bool:
        return self.final_answer is not None


def parse_llm_output(text: str) -> ParsedStep:
    thought_match = re.search(r"Thought:\s*(.+?)(?=\n(?:Action|Final Answer):|$)", text, re.DOTALL)
    thought = thought_match.group(1).strip() if thought_match else None
    final_match = FINAL_RE.search(text)
    if final_match:
        return ParsedStep(thought, None, None, final_match.group(1).strip(), text)
    action_match = ACTION_RE.search(text)
    if action_match:
        return ParsedStep(thought, action_match.group(1), action_match.group(2).strip(), None, text)
    return ParsedStep(thought, None, None, None, text)


# Sanity check the parser.
_example = (
    "Thought: I need to compute 2 + 2.\n"
    "Action: calculator[2 + 2]\n"
)
ps = parse_llm_output(_example)
print(ps)


In [ ]:
# Safe calculator: AST walk instead of eval. Supports ints, floats,
# +, -, *, /, **, parentheses, unary minus.

_ALLOWED_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.UAdd: op.pos,
    ast.Mod: op.mod,
}


def _eval_ast(node: ast.AST) -> float:
    if isinstance(node, ast.Num):
        return node.n
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError(f"unsupported constant {node.value!r}")
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.left), _eval_ast(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.operand))
    raise ValueError(f"unsupported node {type(node).__name__}")


def calculator(expression: str) -> str:
    try:
        tree = ast.parse(expression.strip(), mode="eval")
        value = _eval_ast(tree.body)
    except Exception as e:  # noqa: BLE001 - tool outputs must not raise
        return f"ERROR: {type(e).__name__}: {e}"
    if isinstance(value, float):
        if value.is_integer():
            return str(int(value))
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


# A tiny "world" of 25 stubbed facts for wiki_search.
FACTS: dict[str, str] = {
    "capital of france": "Paris",
    "capital of japan": "Tokyo",
    "capital of brazil": "Brasilia",
    "capital of canada": "Ottawa",
    "capital of kenya": "Nairobi",
    "capital of australia": "Canberra",
    "capital of south africa": "Pretoria",
    "capital of egypt": "Cairo",
    "capital of argentina": "Buenos Aires",
    "capital of mexico": "Mexico City",
    "tallest mountain": "Mount Everest at 8848 metres.",
    "longest river": "The Nile is 6650 km long.",
    "speed of light": "Approximately 299,792,458 metres per second in vacuum.",
    "electron charge": "Approximately 1.602e-19 coulombs.",
    "planck constant": "Approximately 6.626e-34 joule-seconds.",
    "number of continents": "Seven.",
    "chemical symbol for gold": "Au.",
    "chemical symbol for iron": "Fe.",
    "chemical symbol for silver": "Ag.",
    "chemical symbol for tungsten": "W.",
    "author of hamlet": "William Shakespeare.",
    "author of the iliad": "Homer.",
    "author of war and peace": "Leo Tolstoy.",
    "author of the great gatsby": "F. Scott Fitzgerald.",
    "painter of the mona lisa": "Leonardo da Vinci.",
}


def wiki_search(query: str) -> str:
    q = query.strip().lower().rstrip("?").strip()
    # Exact match first.
    if q in FACTS:
        return FACTS[q]
    # Substring fallback.
    for key, val in FACTS.items():
        if key in q or q in key:
            return val
    return "No matching result in the local knowledge base."


# Frozen "now" so the notebook is deterministic.
FIXED_NOW = datetime(2026, 4, 17, 12, 0, tzinfo=timezone.utc)


def get_datetime(spec: str) -> str:
    spec = spec.strip().lower()
    if spec in {"", "now", "today", "current date"}:
        return FIXED_NOW.date().isoformat()
    if spec == "year":
        return str(FIXED_NOW.year)
    if spec == "month":
        return FIXED_NOW.strftime("%B")
    if spec == "day":
        return FIXED_NOW.strftime("%A")
    if spec == "tomorrow":
        return (FIXED_NOW + timedelta(days=1)).date().isoformat()
    if spec == "yesterday":
        return (FIXED_NOW - timedelta(days=1)).date().isoformat()
    return f"ERROR: unknown date spec {spec!r}"


TOOLS: dict[str, callable] = {
    "calculator": calculator,
    "wiki_search": wiki_search,
    "get_datetime": get_datetime,
}
print("tools:", list(TOOLS))


In [ ]:
# A deterministic rule-based policy that emits ReAct-format outputs. It's a
# stand-in for a real LLM so that the notebook runs in CI without any
# model download. Swap in a real LLM via VLLMPolicy below when running on a
# machine with vLLM/Ollama installed.


class Policy:
    def step(self, prompt: str, scratch: str) -> str:
        raise NotImplementedError


class RuleBasedPolicy(Policy):
    '''Routes each question to the right tool via simple keyword matching.'''

    NUMERIC_CHARS = set("0123456789+-*/%().^ ")

    def step(self, prompt: str, scratch: str) -> str:
        # If we already called a tool, the observation is in the scratch.
        # Use that directly as the final answer.
        if "Observation:" in scratch:
            last_obs = scratch.rsplit("Observation:", 1)[1].strip()
            return f"Thought: I have the answer now.\nFinal Answer: {last_obs}"
        q = prompt.lower()
        # Arithmetic: mostly digits and operators, or starts with compute/evaluate.
        if any(w in q for w in ("compute", "evaluate", "what is", "calculate", "sum", "product")):
            expr = re.sub(r"[^0-9+\-*/%().^ \t]", " ", prompt).strip()
            if expr:
                return f"Thought: I need a calculator.\nAction: calculator[{expr}]"
        # Temporal.
        if any(w in q for w in ("today", "current", "date", "what year", "what month", "day of the week", "tomorrow", "yesterday")):
            if "year" in q:
                arg = "year"
            elif "month" in q:
                arg = "month"
            elif "day of" in q:
                arg = "day"
            elif "tomorrow" in q:
                arg = "tomorrow"
            elif "yesterday" in q:
                arg = "yesterday"
            else:
                arg = "today"
            return f"Thought: I need the current date.\nAction: get_datetime[{arg}]"
        # Default to wiki_search with the question as-is.
        return f"Thought: I should look this up.\nAction: wiki_search[{prompt.rstrip('?').strip()}]"


class VLLMPolicy(Policy):
    '''Optional: use a local vLLM endpoint serving Qwen2.5-0.5B-Instruct.

    Set VLLM_URL to your OpenAI-compatible endpoint, e.g.
    ``http://localhost:8000/v1/chat/completions``. Raises SystemExit if the
    endpoint is unreachable so the notebook remains deterministic under CI.
    '''

    def __init__(self, url: str, model: str) -> None:
        self.url = url
        self.model = model

    def step(self, prompt: str, scratch: str) -> str:
        import urllib.request, json
        messages = [
            {"role": "system", "content": (
                "You are a ReAct agent. Respond with 'Thought: ...' and either "
                "'Action: calculator[expr]', 'Action: wiki_search[query]', "
                "'Action: get_datetime[today|year|month|day|tomorrow|yesterday]', "
                "or 'Final Answer: ...'."
            )},
            {"role": "user", "content": prompt + "\n" + scratch},
        ]
        req = urllib.request.Request(
            self.url,
            data=json.dumps({"model": self.model, "messages": messages, "temperature": 0}).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read())
        return data["choices"][0]["message"]["content"]


In [ ]:
@dataclass
class EpisodeResult:
    task: str
    expected: str
    final: str | None
    steps: int
    parse_errors: int
    transcript: str


def run_episode(task: str, expected: str, policy: Policy, max_steps: int = 6) -> EpisodeResult:
    scratch = ""
    parse_errors = 0
    for step in range(max_steps):
        out = policy.step(task, scratch)
        parsed = parse_llm_output(out)
        if parsed.is_final:
            return EpisodeResult(task, expected, parsed.final_answer, step + 1, parse_errors, scratch + out)
        if not parsed.is_action:
            parse_errors += 1
            scratch += f"\n{out}\nObservation: parse error; expected an Action or Final Answer.\n"
            continue
        tool = TOOLS.get(parsed.action)
        if tool is None:
            parse_errors += 1
            scratch += f"\n{out}\nObservation: unknown tool {parsed.action!r}\n"
            continue
        obs = tool(parsed.action_input or "")
        scratch += f"\n{out}\nObservation: {obs}\n"
    return EpisodeResult(task, expected, None, max_steps, parse_errors, scratch)


def grade(result: EpisodeResult) -> bool:
    if result.final is None:
        return False
    return result.expected.lower() in result.final.lower()


In [ ]:
TASKS: list[tuple[str, str]] = [
    # 8 arithmetic
    ("Compute 12 + 30", "42"),
    ("What is 7 * 8?", "56"),
    ("Calculate (15 + 5) / 4", "5"),
    ("Compute 2 ** 10", "1024"),
    ("Evaluate 100 - 37", "63"),
    ("What is 1234 + 5678?", "6912"),
    ("Compute 9 * 9 - 1", "80"),
    ("Calculate 144 / 12", "12"),
    # 8 lookup
    ("What is the capital of France?", "Paris"),
    ("What is the capital of Japan?", "Tokyo"),
    ("What is the tallest mountain?", "Everest"),
    ("Who wrote Hamlet?", "Shakespeare"),
    ("What is the chemical symbol for gold?", "Au"),
    ("What is the speed of light?", "299,792,458"),
    ("Who painted the Mona Lisa?", "Leonardo"),
    ("How many continents are there?", "Seven"),
    # 4 temporal
    ("What is the current year?", "2026"),
    ("What is the current month?", "April"),
    ("What is the date tomorrow?", "2026-04-18"),
    ("What is the date yesterday?", "2026-04-16"),
]

policy = RuleBasedPolicy()
results: list[EpisodeResult] = []
for task, expected in TASKS:
    results.append(run_episode(task, expected, policy))

n_success = sum(grade(r) for r in results)
total_parse_errors = sum(r.parse_errors for r in results)
avg_steps = sum(r.steps for r in results) / len(results)
success_rate = n_success / len(results)
parse_err_rate = total_parse_errors / sum(r.steps for r in results)

for r in results[:6]:
    print(f"{r.task:<45} expected={r.expected!r:<20} final={r.final!r}")
print(f"\nsuccess {n_success}/{len(results)} ({success_rate:.0%}); "
      f"avg steps {avg_steps:.2f}; parse-error rate {parse_err_rate:.1%}")


In [ ]:
s.check(
    "success_rate_at_least_70pct",
    lambda: success_rate >= 0.70,
    msg=f"success_rate={success_rate:.2%}",
)
s.check(
    "parse_error_rate_at_most_10pct",
    lambda: parse_err_rate <= 0.10,
    msg=f"parse_err_rate={parse_err_rate:.2%}",
)
s.check(
    "avg_steps_at_most_5",
    lambda: avg_steps <= 5.0,
    msg=f"avg_steps={avg_steps:.2f}",
)
s.check(
    "every_task_has_finite_trace",
    lambda: all(r.steps >= 1 for r in results),
    msg="each episode must step at least once",
)
s.check(
    "parser_extracts_action_from_well_formed_output",
    lambda: parse_llm_output("Thought: x\nAction: calculator[2+2]").action == "calculator",
    msg="parser smoke test",
)


In [ ]:
s.summary()
s.save()
